# Q1: Which past indicators predict future KSI crashes?

Builds a validated KSI risk score for every crash site (issue #19). All logic lives in the layers; this notebook only wires them together.

**Design decisions** (each one is set in config):
- **Sites:** peak-radius clusters of geocoded crash locations (every location within 100 ft of the busiest unclaimed one, so sites cannot chain along corridors), built only from crashes before 2024 (`SiteConfig`).
- **Outcome:** KSI crashes per site-year. KSI means a death, or a severe-injury proxy from the persons table's `complaint`/`emotional_status`, because public data has no "seriously injured" field (`CrashConfig`).
- **Predictors:** 3-year lagged counts of KSI, fatal, injury and all crashes and of pedestrian and cyclist casualties, entered as `log1p`, plus highway share and camera flags (`PanelConfig`).
- **Validation:** expanding window. For each test year (2024, 2025), fit on earlier years and score that year. Metrics are reported for all sites and for non-highway sites (`ValidationConfig`).

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("..")

import pandas as pd

from model.count_models import HistoryRanker, NegativeBinomialModel
from preprocessing.panel import load_panel
from validation.validate import ValidationConfig, evaluate

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.precision", 3)

In [2]:
panel, scoring, sites = load_panel()
panel.shape, scoring.shape, sites.shape

clean_crashes: dropped 0 crashes before 2017-01-01


drop_ungeocoded: dropped 95967 of 1311313 crashes (7.3%) with missing or out-of-NYC coordinates


flag_highway: 146081 of 1215346 crashes flagged as highway


flag_ksi: 25873 KSI crashes (2107 with a death); 3533 crashes had no person records


build_site_points: 1013673 crashes before 2024-01-01 at 210488 unique points -> 89073 sites


assign_sites: 18031 crashes unassigned (18031 of 201673 on/after 2024-01-01)


count_site_years: 15 crashes with missing injury counts treated as 0


assign_cameras: 4308 of 4610 cameras matched to a site within 150 ft


load_panel: 534438 site-years for 89073 sites


((534438, 24), (89073, 24), (89073, 8))

## Model ladder
Rung 1 is current practice (rank by past KSI). If a crash-history model ranks better, past KSI alone is too sparse a signal.

In [3]:
LAGS = [c for c in panel if c.startswith("log1p_")]
CONTEXT = ["highway_share", "rlc_active", "speed_active"]

ladder = {
    "1 past KSI (baseline)": lambda: HistoryRanker("ksi_lag3", n_years=3),
    "2 past injury crashes": lambda: HistoryRanker("injury_lag3", n_years=3),
    "3 NB: KSI history": lambda: NegativeBinomialModel(["log1p_ksi_lag3"]),
    "4 NB: crash history": lambda: NegativeBinomialModel(LAGS),
    "5 NB: + highway, cameras": lambda: NegativeBinomialModel(LAGS + CONTEXT),
}
config = ValidationConfig()

results, calibrations = [], []
for name, make_model in ladder.items():
    metrics, calibration, _ = evaluate(make_model, panel, config)
    results.append(metrics.assign(model=name))
    calibrations.append(calibration.assign(model=name))

# Sensitivity: drop every site that ever had a camera
no_camera = panel[~panel["camera_site"]]
for name in ["1 past KSI (baseline)", "4 NB: crash history"]:
    metrics, _, _ = evaluate(ladder[name], no_camera, config)
    results.append(metrics.assign(model=f"no-camera sites: {name}"))

results = pd.concat(results)
calibrations = pd.concat(calibrations)

### Share of next year's KSI crashes captured by the top-k sites (mean of test years)

In [4]:
(
    results[results["metric"] == "top_k_capture"]
    .pivot_table(index=["scope", "model"], columns="k", values="value")
)

k                                                   50.0    100.0   250.0   \
scope       model                                                            
all         1 past KSI (baseline)                    0.012   0.020   0.040   
            2 past injury crashes                    0.013   0.025   0.057   
            3 NB: KSI history                        0.012   0.020   0.040   
            4 NB: crash history                      0.013   0.026   0.055   
            5 NB: + highway, cameras                 0.013   0.026   0.057   
            no-camera sites: 1 past KSI (baseline)   0.011   0.018   0.036   
            no-camera sites: 4 NB: crash history     0.014   0.029   0.058   
non_highway 1 past KSI (baseline)                    0.013   0.022   0.043   
            2 past injury crashes                    0.015   0.029   0.057   
            3 NB: KSI history                        0.013   0.022   0.043   
            4 NB: crash history                      0.015   0.029   0.058   
            5 NB: + highway, cameras                 0.015   0.029   0.058   
            no-camera sites: 1 past KSI (baseline)   0.012   0.019   0.036   
            no-camera sites: 4 NB: crash history     0.017   0.027   0.056   

k                                                   500.0   1000.0  
scope       model                                                   
all         1 past KSI (baseline)                    0.064   0.110  
            2 past injury crashes                    0.095   0.160  
            3 NB: KSI history                        0.064   0.110  
            4 NB: crash history                      0.096   0.157  
            5 NB: + highway, cameras                 0.097   0.158  
            no-camera sites: 1 past KSI (baseline)   0.059   0.101  
            no-camera sites: 4 NB: crash history     0.097   0.154  
non_highway 1 past KSI (baseline)                    0.068   0.115  
            2 past injury crashes                    0.097   0.159  
            3 NB: KSI history                        0.068   0.115  
            4 NB: crash history                      0.100   0.161  
            5 NB: + highway, cameras                 0.098   0.161  
            no-camera sites: 1 past KSI (baseline)   0.061   0.097  
            no-camera sites: 4 NB: crash history     0.094   0.150

### Fit: Poisson deviance (lower is better) and predicted / observed total KSI

In [5]:
(
    results[results["metric"] != "top_k_capture"]
    .pivot_table(index=["scope", "model"], columns="metric", values="value")
)

metric                                              poisson_deviance  \
scope       model                                                      
all         1 past KSI (baseline)                              0.568   
            2 past injury crashes                              0.846   
            3 NB: KSI history                                  0.196   
            4 NB: crash history                                0.162   
            5 NB: + highway, cameras                           0.162   
            no-camera sites: 1 past KSI (baseline)             0.531   
            no-camera sites: 4 NB: crash history               0.152   
non_highway 1 past KSI (baseline)                              0.527   
            2 past injury crashes                              0.778   
            3 NB: KSI history                                  0.184   
            4 NB: crash history                                0.152   
            5 NB: + highway, cameras                           0.152   
            no-camera sites: 1 past KSI (baseline)             0.490   
            no-camera sites: 4 NB: crash history               0.143   

metric                                              pred_to_obs_total  
scope       model                                                      
all         1 past KSI (baseline)                               1.003  
            2 past injury crashes                              13.734  
            3 NB: KSI history                                   1.030  
            4 NB: crash history                                 1.014  
            5 NB: + highway, cameras                            1.014  
            no-camera sites: 1 past KSI (baseline)              1.001  
            no-camera sites: 4 NB: crash history                1.015  
non_highway 1 past KSI (baseline)                               1.027  
            2 past injury crashes                              13.708  
            3 NB: KSI history                                   1.109  
            4 NB: crash history                                 1.041  
            5 NB: + highway, cameras                            1.046  
            no-camera sites: 1 past KSI (baseline)              1.024  
            no-camera sites: 4 NB: crash history                1.046

### Calibration of the crash-history model by decile of predicted risk (all sites)

In [6]:
calibrations.query("model == '4 NB: crash history' and scope == 'all'").pivot_table(
    index="bin", columns="test_year", values=["mean_pred", "mean_obs"]
)

mean_obs        mean_pred       
test_year     2024   2025      2024   2025
bin                                       
1            0.009  0.010     0.007  0.006
2            0.003  0.004     0.007  0.006
3            0.003  0.002     0.007  0.006
4            0.007  0.009     0.011  0.008
5            0.003  0.004     0.011  0.010
6            0.011  0.012     0.014  0.013
7            0.015  0.012     0.018  0.017
8            0.024  0.026     0.026  0.025
9            0.056  0.053     0.045  0.043
10           0.170  0.157     0.161  0.155

## Which indicators carry the signal?
Rate ratios from the crash-history model fit on every year. Standard errors are clustered by site. The lag counts are nested (KSI ⊂ injury ⊂ all crashes), so read each coefficient as the effect *given the others*.

In [7]:
final = NegativeBinomialModel(LAGS).fit(panel, panel["ksi"], groups=panel["site_id"])
final.coefficients()

,coef,rate_ratio,ci_low,ci_high,p_value
const,-5.509e+00,0.004,0.004,0.004,0.000e+00
log1p_ksi_lag3,-3.263e-02,0.968,0.923,1.015,1.799e-01
log1p_fatal_lag3,-2.370e-02,0.977,0.862,1.106,7.090e-01
log1p_injury_lag3,4.455e-01,1.561,1.493,1.633,3.607e-85
log1p_crashes_lag3,7.277e-01,2.070,1.997,2.147,0.000e+00
log1p_ped_injured_lag3,9.523e-02,1.100,1.066,1.135,3.935e-09
log1p_cyc_injured_lag3,1.690e-01,1.184,1.143,1.227,2.203e-20
year_2021,-3.343e-04,1.000,0.947,1.056,9.904e-01
year_2022,1.762e-01,1.193,1.128,1.261,6.425e-10
year_2023,4.385e-01,1.550,1.465,1.641,1.772e-51


## Risk score for every site
Expected KSI crashes in the year after the data ends, from the model above (the latest year's fixed effect is carried forward). The file is written to `experiments/outputs/`, which git ignores.

In [8]:
risk = (
    scoring[["site_id", "year", "ksi_lag3", "injury_lag3", "crashes_lag3", "highway_share", "camera_site"]]
    .assign(expected_ksi=final.predict(scoring))
    .merge(sites[["site_id", "latitude", "longitude", "n_points", "span_ft"]], on="site_id")
    .sort_values("expected_ksi", ascending=False)
    .assign(rank=lambda d: range(1, len(d) + 1))
)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
risk.to_parquet(out_dir / f"q1_site_risk_{risk['year'].iloc[0]}.parquet")
risk.head(25)

,site_id,year,ksi_lag3,injury_lag3,crashes_lag3,highway_share,camera_site,expected_ksi,latitude,longitude,n_points,span_ft,rank
0,0,2026,6.0,83.0,172.0,0.298,False,2.587,40.862,-73.913,3,17.856,1
19,19,2026,2.0,72.0,143.0,0.515,True,2.382,40.668,-73.926,10,79.056,2
3,3,2026,1.0,57.0,121.0,0.000,True,1.952,40.676,-73.897,6,73.700,3
1,1,2026,1.0,73.0,195.0,1.000,False,1.914,40.609,-74.038,1,0.000,4
4694,4694,2026,4.0,59.0,114.0,0.255,False,1.806,40.805,-73.912,8,131.246,5
272,272,2026,2.0,63.0,119.0,0.000,False,1.776,40.813,-73.898,4,101.749,6
5,5,2026,2.0,59.0,109.0,0.000,True,1.749,40.659,-73.891,7,57.804,7
73,73,2026,2.0,91.0,153.0,0.795,False,1.746,40.711,-73.728,4,44.835,8
175,175,2026,3.0,67.0,121.0,0.008,True,1.706,40.816,-73.895,3,65.355,9
86,86,2026,3.0,53.0,131.0,0.573,False,1.647,40.762,-73.757,6,54.389,10
